# Osa 2: Puheentunnistus Whisper-mallilla

## Asennus ja kirjastot

In [ ]:
%pip install -q openai-whisper librosa jiwer pandas matplotlib

In [ ]:
import time
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import whisper
import jiwer
from IPython.display import Audio, display

laite = "cuda" if torch.cuda.is_available() else "cpu"
print("Laite:", laite)

## Ääninäytteet

In [ ]:
# viite = oikea litterointi (täytetään käsin, tarvitaan WER-laskentaan)
# kieli = "fi" tai "en"
NAYTTEET = [
    {"nimi": "selkea_fi",  "tiedosto": "aanet/selkea_fi.wav",  "kieli": "fi", "kuvaus": "", "viite": ""},
    {"nimi": "hali_fi",    "tiedosto": "aanet/hali_fi.wav",    "kieli": "fi", "kuvaus": "", "viite": ""},
    {"nimi": "puhuja2_fi", "tiedosto": "aanet/puhuja2_fi.wav", "kieli": "fi", "kuvaus": "", "viite": ""},
    {"nimi": "murre_fi",   "tiedosto": "aanet/murre_fi.wav",   "kieli": "fi", "kuvaus": "", "viite": ""},
    {"nimi": "selkea_en",  "tiedosto": "aanet/selkea_en.wav",  "kieli": "en", "kuvaus": "", "viite": ""},
]

MALLIT = ["tiny", "base", "small"]

In [ ]:
for n in NAYTTEET:
    print(n["nimi"], "-", n["tiedosto"])
    display(Audio(n["tiedosto"]))

## Mel-spektrogrammi

In [ ]:
def piirra_spektrogrammi(tiedosto, otsikko=""):
    audio = whisper.load_audio(tiedosto)  # 16 kHz mono
    mel = whisper.log_mel_spectrogram(audio).cpu().numpy()  # (80, kehykset)
    kesto = len(audio) / whisper.audio.SAMPLE_RATE

    fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    ax[0].plot(np.linspace(0, kesto, len(audio)), audio, linewidth=0.5)
    ax[0].set_ylabel("Amplitudi")
    ax[0].set_title(f"Aaltomuoto - {otsikko}")

    kuva = ax[1].imshow(mel, aspect="auto", origin="lower", extent=[0, kesto, 0, mel.shape[0]])
    ax[1].set_xlabel("Aika (s)")
    ax[1].set_ylabel("Mel-kaista")
    ax[1].set_title("Log-mel-spektrogrammi")
    fig.colorbar(kuva, ax=ax[1], label="Log-intensiteetti (dB-tyyppinen)")
    plt.tight_layout()
    plt.show()
    print("Mel-matriisin muoto:", mel.shape)

piirra_spektrogrammi(NAYTTEET[0]["tiedosto"], NAYTTEET[0]["nimi"])

In [ ]:
for n in NAYTTEET[1:]:
    piirra_spektrogrammi(n["tiedosto"], n["nimi"])

## Mallien lataus

In [ ]:
mallit = {}
lataus_ajat = {}
for nimi in MALLIT:
    t0 = time.perf_counter()
    mallit[nimi] = whisper.load_model(nimi, device=laite)
    lataus_ajat[nimi] = time.perf_counter() - t0
    parametrit = sum(p.numel() for p in mallit[nimi].parameters())
    print(f"{nimi}: {parametrit/1e6:.0f} M parametria, latausaika {lataus_ajat[nimi]:.1f} s")

## Tunnistus

In [ ]:
def tunnista(malli_nimi, tiedosto, kieli):
    t0 = time.perf_counter()
    tulos = mallit[malli_nimi].transcribe(tiedosto, language=kieli, fp16=(laite == "cuda"))
    return tulos["text"].strip(), time.perf_counter() - t0

rivit = []
for n in NAYTTEET:
    for m in MALLIT:
        teksti, aika = tunnista(m, n["tiedosto"], n["kieli"])
        rivit.append({"nayte": n["nimi"], "malli": m, "aika_s": aika, "tunnistettu": teksti})
        print(f"[{m:>5}] {n['nimi']}: {teksti}")

tulokset = pd.DataFrame(rivit)

## Vertailu: tarkkuus

In [ ]:
def normalisoi(s):
    s = s.lower()
    s = re.sub(r"[^\w\s]", "", s)
    return re.sub(r"\s+", " ", s).strip()

viitteet = {n["nimi"]: n["viite"] for n in NAYTTEET}

def laske_wer(rivi):
    viite = viitteet[rivi["nayte"]]
    if not viite:
        return np.nan
    return jiwer.wer(normalisoi(viite), normalisoi(rivi["tunnistettu"]))

def laske_cer(rivi):
    viite = viitteet[rivi["nayte"]]
    if not viite:
        return np.nan
    return jiwer.cer(normalisoi(viite), normalisoi(rivi["tunnistettu"]))

tulokset["WER"] = tulokset.apply(laske_wer, axis=1)
tulokset["CER"] = tulokset.apply(laske_cer, axis=1)
tulokset

In [ ]:
display(tulokset.pivot(index="nayte", columns="malli", values="WER")[MALLIT])
display(tulokset.pivot(index="nayte", columns="malli", values="CER")[MALLIT])

In [ ]:
tulokset.pivot(index="nayte", columns="malli", values="WER")[MALLIT].plot.bar(figsize=(10, 4))
plt.ylabel("WER")
plt.title("Sanavirheprosentti malleittain")
plt.tight_layout()
plt.show()

## Vertailu: nopeus

In [ ]:
kestot = {n["nimi"]: whisper.load_audio(n["tiedosto"]).shape[0] / whisper.audio.SAMPLE_RATE for n in NAYTTEET}
tulokset["kesto_s"] = tulokset["nayte"].map(kestot)
tulokset["RTF"] = tulokset["aika_s"] / tulokset["kesto_s"]  # alle 1 = nopeampi kuin reaaliaika

display(tulokset.pivot(index="nayte", columns="malli", values="aika_s")[MALLIT])
display(tulokset.groupby("malli")[["aika_s", "RTF"]].mean().loc[MALLIT])

In [ ]:
tulokset.pivot(index="nayte", columns="malli", values="aika_s")[MALLIT].plot.bar(figsize=(10, 4))
plt.ylabel("Aika (s)")
plt.title("Tunnistusaika malleittain")
plt.tight_layout()
plt.show()

## Tunnistetut tekstit rinnakkain

In [ ]:
pd.set_option("display.max_colwidth", None)
for n in NAYTTEET:
    print("=" * 80)
    print(n["nimi"], "-", n["kuvaus"])
    print("VIITE:", n["viite"])
    for m in MALLIT:
        r = tulokset[(tulokset.nayte == n["nimi"]) & (tulokset.malli == m)].iloc[0]
        print(f"{m:>6}: {r['tunnistettu']}")

## Kohinakoe

In [ ]:
def lisaa_kohinaa(audio, snr_db):
    teho_signaali = np.mean(audio ** 2)
    teho_kohina = teho_signaali / (10 ** (snr_db / 10))
    kohina = np.random.normal(0, np.sqrt(teho_kohina), audio.shape)
    return (audio + kohina).astype(np.float32)

KOHINA_NAYTE = NAYTTEET[0]
SNR_ARVOT = [20, 10, 5, 0, -5]

audio_alku = whisper.load_audio(KOHINA_NAYTE["tiedosto"])
kohina_rivit = []
np.random.seed(0)
for snr in SNR_ARVOT:
    meluisa = lisaa_kohinaa(audio_alku, snr)
    for m in MALLIT:
        t0 = time.perf_counter()
        teksti = mallit[m].transcribe(meluisa, language=KOHINA_NAYTE["kieli"], fp16=(laite == "cuda"))["text"].strip()
        aika = time.perf_counter() - t0
        wer = jiwer.wer(normalisoi(KOHINA_NAYTE["viite"]), normalisoi(teksti)) if KOHINA_NAYTE["viite"] else np.nan
        kohina_rivit.append({"SNR_dB": snr, "malli": m, "WER": wer, "aika_s": aika, "tunnistettu": teksti})

kohina_tulokset = pd.DataFrame(kohina_rivit)
kohina_tulokset

In [ ]:
kohina_tulokset.pivot(index="SNR_dB", columns="malli", values="WER")[MALLIT].sort_index(ascending=False).plot(marker="o", figsize=(8, 4))
plt.xlabel("SNR (dB)")
plt.ylabel("WER")
plt.title("Kohinan vaikutus tunnistustarkkuuteen")
plt.gca().invert_xaxis()
plt.show()

In [ ]:
import soundfile as sf
sf.write("aanet/kohina_0dB.wav", lisaa_kohinaa(audio_alku, 0), whisper.audio.SAMPLE_RATE)
piirra_spektrogrammi("aanet/kohina_0dB.wav", "SNR 0 dB")

## Kieli ja englanninkielinen puhe

In [ ]:
# Kielen automaattinen tunnistus: kieltä ei anneta mallille
rivit = []
for n in NAYTTEET:
    for m in MALLIT:
        tulos = mallit[m].transcribe(n["tiedosto"], language=None, fp16=(laite == "cuda"))
        rivit.append({"nayte": n["nimi"], "malli": m, "oikea_kieli": n["kieli"], "tunnistettu_kieli": tulos["language"]})
pd.DataFrame(rivit).pivot(index=["nayte", "oikea_kieli"], columns="malli", values="tunnistettu_kieli")[MALLIT]

In [ ]:
display(tulokset[tulokset.nayte.str.endswith("_en")][["nayte", "malli", "aika_s", "WER", "CER", "tunnistettu"]])
display(tulokset[tulokset.nayte.str.endswith("_fi")].groupby("malli")[["WER", "CER"]].mean().loc[MALLIT])

## Segmentit ja aikaleimat

In [ ]:
tulos = mallit[MALLIT[-1]].transcribe(NAYTTEET[0]["tiedosto"], language="fi", fp16=(laite == "cuda"))
for s in tulos["segments"]:
    print(f"[{s['start']:6.2f} - {s['end']:6.2f}]  {s['text'].strip()}  (avg_logprob={s['avg_logprob']:.2f}, no_speech={s['no_speech_prob']:.2f})")

## Yhteenveto

In [ ]:
yhteenveto = tulokset.groupby("malli")[["WER", "CER", "aika_s", "RTF"]].mean().loc[MALLIT]
yhteenveto